In [1]:
import os
import pandas as pd
import pyarrow.parquet as pq

In [2]:
def load_all_parquets():
    bronze_data_path = os.path.join("raw_eph_db")
    dfs = {}
    for root, dirs, files in os.walk(bronze_data_path):
        print(f"Found {len(files)} files in {root}")
        for file in files:
            if file.endswith(".parquet") and not file.startswith("base_ind_2011_3"):
                file_path = os.path.join(root, file)
                print(f"Loading {file_path}...")
                try:
                    table = pq.read_table(file_path)
                    df = table.to_pandas()
                    print(f"Loaded {len(df)} rows from {file_path}")
                    dfs.update({file_path: df})
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    # Optionally, try to fix invalid UTF-8
                    try:
                        import pyarrow as pa
                        import pyarrow.compute as pc
                        table = pq.read_table(file_path)
                        # For each string column, replace invalid UTF-8
                        schema = table.schema
                        new_columns = []
                        for i, field in enumerate(schema):
                            col = table.column(i)
                            if pa.types.is_string(field.type) or pa.types.is_large_string(field.type):
                                # Replace invalid UTF-8 with replacement character
                                col = pc.utf8_replace_invalid(col, replacement='�')
                            new_columns.append(col)
                        fixed_table = pa.Table.from_arrays(new_columns, names=schema.names)
                        df = fixed_table.to_pandas()
                        print(f"Loaded {len(df)} rows from {file_path} after fixing UTF-8")
                        dfs.update({file_path: df})
                    except Exception as e2:
                        print(f"Failed to fix {file_path}: {e2}")
    return dfs

In [9]:
def load_all_csvs():
    bronze_data_path = os.path.join("raw_eph_db")
    dfs = {}
    for root, dirs, files in os.walk(bronze_data_path):
        print(f"Found {len(files)} files in {root}")
        for file in files:
            if file.endswith(".csv"):
                file_path = os.path.join(root, file)
                print(f"Loading {file_path}...")
                try:
                    table = pd.read_csv(file_path, low_memory=False, encoding="latin1")
                    print(f"Loaded {len(table)} rows from {file_path}")
                    dfs.update({file_path: table})
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    return dfs

In [10]:
dfs = load_all_csvs()

Found 200 files in raw_eph_db
Loading raw_eph_db\base_ind_1996_1.csv...
Loaded 113209 rows from raw_eph_db\base_ind_1996_1.csv
Loading raw_eph_db\base_ind_1996_2.csv...
Loaded 107692 rows from raw_eph_db\base_ind_1996_2.csv
Loading raw_eph_db\base_ind_1997_1.csv...
Loaded 110487 rows from raw_eph_db\base_ind_1997_1.csv
Loading raw_eph_db\base_ind_1997_2.csv...
Loaded 109302 rows from raw_eph_db\base_ind_1997_2.csv
Loading raw_eph_db\base_ind_1998_1.csv...
Loaded 105476 rows from raw_eph_db\base_ind_1998_1.csv
Loading raw_eph_db\base_ind_1998_2.csv...
Loaded 99174 rows from raw_eph_db\base_ind_1998_2.csv
Loading raw_eph_db\base_ind_1999_1.csv...
Loaded 92294 rows from raw_eph_db\base_ind_1999_1.csv
Loading raw_eph_db\base_ind_1999_2.csv...
Loaded 91707 rows from raw_eph_db\base_ind_1999_2.csv
Loading raw_eph_db\base_ind_2000_1.csv...
Loaded 83630 rows from raw_eph_db\base_ind_2000_1.csv
Loading raw_eph_db\base_ind_2000_2.csv...
Loaded 83399 rows from raw_eph_db\base_ind_2000_2.csv
Loadi

In [11]:
# Build a new DataFrame that has the name of the file in the row (without the extension) and the name of the columns as columns. The value of the cell is a flag, indicating whether the column is present in the file or not.
all_columns = sorted(set(col for df in dfs.values() for col in df.columns))
meta_df = pd.DataFrame(columns=["file"] + all_columns)
rows = []
for file_path, df in dfs.items():
    file_name = os.path.basename(file_path).replace(".parquet", "")
    row = {"file": file_name}
    for col in all_columns:
        row[col] = 1 if col in df.columns else 0
    rows.append(row)
meta_df = pd.DataFrame(rows)

In [12]:
meta_df.to_clipboard()